### Evaluation and Metrics Module

In [ ]:
# imports
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [ ]:
class Evaluator:
    def __init__(self, model, device=None):
        self.model = model
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def evaluate_accuracy(self, dataloader):
        self.model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in dataloader:
                data = data.to(self.device)
                target = target.to(self.device)

                output = self.model(data)
                preds = output.argmax(dim=1)

                correct += (preds == target).sum().item()
                total += target.size(0)

        return correct / total

    def get_predictions(self, dataloader):
        self.model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for data, target in dataloader:
                data = data.to(self.device)
                target = target.to(self.device)

                output = self.model(data)
                preds = output.argmax(dim=1)

                all_preds.append(preds.cpu())
                all_labels.append(target.cpu())

        return torch.cat(all_preds), torch.cat(all_labels)

    def confusion_matrix(self, dataloader, num_classes=10):
        preds, labels = self.get_predictions(dataloader)

        cm = torch.zeros((num_classes, num_classes), dtype=torch.int32)

        for t, p in zip(labels, preds):
            cm[t, p] += 1

        return cm

## Attack Success Rate

In [ ]:
def targeted_attack_success_rate(model, dataloader, source_class, target_class, device=None):
    device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    total_source = 0
    successful = 0

    with torch.no_grad():
        for data, labels in dataloader:
            data = data.to(device)
            labels = labels.to(device)

            mask = (labels == source_class)
            if mask.sum() == 0:
                continue

            output = model(data)
            preds = output.argmax(dim=1)

            total_source += mask.sum().item()
            successful += (preds[mask] == target_class).sum().item()

    if total_source == 0:
        return 0.0

    return successful / total_source

### Backdoor Attack Success Rate

In [ ]:
def backdoor_attack_success_rate(model, poisoned_loader, target_label, device=None):
    device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    total = 0
    successful = 0

    with torch.no_grad():
        for data, labels in poisoned_loader:
            data = data.to(device)

            output = model(data)
            preds = output.argmax(dim=1)

            total += preds.size(0)
            successful += (preds == target_label).sum().item()

    return successful / total

In [ ]:
def create_dataloader(images, labels, batch_size=512, shuffle=False):
    dataset = TensorDataset(images, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
clean_loader = create_dataloader(test_images, test_labels)

evaluator = Evaluator(trainer.model)

clean_acc = evaluator.evaluate_accuracy(clean_loader)
print("Clean Test Accuracy:", clean_acc)

In [ ]:
poison_loader = create_dataloader(poisoned_images, poisoned_labels)

poison_acc = evaluator.evaluate_accuracy(poison_loader)
print("Poisoned Accuracy:", poison_acc)

In [ ]:
asr = targeted_attack_success_rate(
    trainer.model,
    poison_loader,
    source_class=1,
    target_class=7
)

print("Targeted Attack Success Rate:", asr)

backdoor_loader = create_dataloader(backdoor_images, backdoor_labels)

asr_backdoor = backdoor_attack_success_rate(
    trainer.model,
    backdoor_loader,
    target_label=0
)

print("Backdoor Attack Success Rate:", asr_backdoor)

In [ ]:
def plot_confusion_matrix(cm):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.show()

In [ ]:
cm = evaluator.confusion_matrix(clean_loader)
plot_confusion_matrix(cm)